# Demo 3 · Statisches HTML mit BeautifulSoup

[Startseite](../README.md) · [Alle Demos](README.md) · [Task 1](../tasks/01_statischer_katalog/README.md)

**Ziel:** Du trennst HTTP-Abruf, HTML-Parsing und Datenprüfung, verwendest CSS-Selektoren und relative Links und erkennst die Grenze eines mehrseitigen Abrufs.

Der Standardlauf ruft zwei synthetische Katalogseiten über echtes lokales HTTP ab. Dafür benötigst Du nur den Workshop-Kernel; Selenium wird nicht verwendet. Optional kannst Du ausdrücklich in den Live-Modus wechseln. Der gesamte Ablauf bleibt auf höchstens zwei Seiten begrenzt; Live-Fehler führen nicht automatisch zum Sample-Modus zurück.


## 1. Umgebung und Datenquelle


In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents]
             if (p / "src/webscraping_workshop/checks.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Öffne dieses Notebook innerhalb des vollständig entpackten Repos.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from webscraping_workshop.checks import fixture_server
from contextlib import ExitStack
from datetime import datetime, timezone
from urllib.parse import urljoin, urlsplit
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display


In [2]:
LIVE = False
MAX_PAGES = 2  # Feste Grenze auch im Live-Modus; nicht erhöhen.
MODE = "live" if LIVE else "sample"
RETRIEVED_AT = datetime.now(timezone.utc).isoformat()
FIXTURE_DIR = ROOT / "data/teaching/books"
LIVE_START_URL = "https://books.toscrape.com/"
print("Modus:", MODE, "| Maximal", MAX_PAGES, "Seiten")


Modus: sample | Maximal 2 Seiten


## 2. Felder aus einem Produkt lesen

Ein `article.product_pod` umfasst ein Produkt. Der Titel steht im `title`-Attribut des Links; der sichtbare Text kann gekürzt sein. `urljoin` löst den relativen Produktlink gegen die tatsächlich erreichte Seiten-URL auf.

BeautifulSoup führt kein JavaScript aus. Hier stehen die Daten schon im gelieferten HTML. Wir prüfen Pflichtfelder und das Preisformat, damit ein veränderter Selektor nicht unbemerkt zu einem leeren Export führt.


In [3]:
def extract_basic(soup, page_url):
    articles = soup.select("article.product_pod")
    if not articles:
        raise ValueError("Keine Produktcontainer gefunden: Antwort und Selektor prüfen.")
    records = []
    for article in articles:
        link = article.select_one("h3 a[title][href]")
        price_node = article.select_one(".price_color")
        if link is None or price_node is None:
            raise ValueError("Titel, Produktlink oder Preis fehlt.")
        title = link["title"].strip()
        href = link["href"].strip()
        price_match = re.fullmatch(r"£(\d+\.\d{2})", price_node.get_text(strip=True))
        if not title or not href or price_match is None:
            raise ValueError("Leerer Titel/Link oder unerwartetes GBP-Preisformat.")
        price = float(price_match.group(1))
        if price <= 0:
            raise ValueError("Preis muss positiv sein.")
        records.append({
            "title": title,
            "price_gbp": price,
            "detail_url": urljoin(page_url, href),
            "source_page": page_url,
        })
    return records


## 3. Höchstens zwei Seiten abrufen

Die Schleife prüft HTTP-Status, HTML-Antwort und Quellhost. Ein Set erkennt wiederholte Seiten. Wir folgen `li.next a` und pausieren im Live-Modus vor dem zweiten Abruf.

**Zwei verschiedene Enden:** `no_next_page` bedeutet, dass kein weiterer Link gefunden wurde. `page_limit` bedeutet, dass unser Sicherheitslimit erreicht wurde und eine weitere Seite existiert. Ein fehlender Weiter-Link allein beweist bei unbekannten Websites noch keine Vollständigkeit; zusätzliche erwartete Gesamtzahlen oder Strukturprüfungen helfen.


In [4]:
def collect_catalog(start_url, session, extract_page):
    records, pages, visited = [], [], set()
    current_url = start_url
    origin = urlsplit(start_url).netloc
    stop_reason = None

    for page_number in range(1, MAX_PAGES + 1):
        if current_url in visited:
            raise ValueError("Pagination-Schleife: Eine Seite wurde bereits besucht.")
        if LIVE and page_number > 1:
            time.sleep(1)  # Höfliche Pause vor der zweiten Live-Anfrage.
        response = session.get(current_url, timeout=(5, 15))
        response.raise_for_status()
        if response.status_code != 200:
            raise ValueError(f"Unerwarteter HTTP-Status: {response.status_code}")
        if urlsplit(response.url).netloc != origin:
            raise ValueError("Weiterleitung verlässt den gewählten Quellhost.")
        if "text/html" not in response.headers.get("Content-Type", "").lower():
            raise ValueError("Erwartet wurde ein HTML-Dokument.")
        visited.add(current_url)
        visited.add(response.url)
        soup = BeautifulSoup(response.content, "lxml")
        page_records = extract_page(soup, response.url)
        records.extend(page_records)
        pages.append({
            "url": response.url,
            "status_code": response.status_code,
            "record_count": len(page_records),
            "html": response.content.decode("utf-8"),
        })

        next_container = soup.select_one("li.next")
        next_link = soup.select_one("li.next a[href]")
        if next_container is not None and (next_link is None or not next_link["href"].strip()):
            raise ValueError("Weiter-Navigation vorhanden, aber gültiger Link fehlt.")
        if next_link is None:
            current_url = None
            stop_reason = "no_next_page"
            break
        current_url = urljoin(response.url, next_link["href"])
        next_parts = urlsplit(current_url)
        if next_parts.scheme not in {"http", "https"} or next_parts.netloc != origin:
            raise ValueError("Weiter-Link verlässt den erlaubten HTTP(S)-Quellhost.")
        if current_url in visited:
            raise ValueError("Pagination-Schleife im Weiter-Link erkannt.")
    else:
        stop_reason = "page_limit"
    return records, pages, stop_reason, current_url


In [5]:
with ExitStack() as stack:
    # ExitStack beendet Server und Session auch bei Exceptions.
    session = stack.enter_context(requests.Session())
    session.headers["User-Agent"] = "WebscrapingWorkshop/0.2 (educational exercise)"
    if LIVE:
        start_url = LIVE_START_URL
    else:
        session.trust_env = False  # Loopback soll keinen Firmenproxy verwenden.
        base_url = stack.enter_context(fixture_server(FIXTURE_DIR))
        start_url = urljoin(base_url + "/", "page-1.html")
    records, pages, stop_reason, remaining_url = collect_catalog(start_url, session, extract_basic)

books = pd.DataFrame(records)
print("Seiten:", len(pages), "| Beobachtungen:", len(books))
print("Eindeutige Produkt-URLs:", books["detail_url"].nunique())
print("Abbruchgrund:", stop_reason)
if remaining_url is not None:
    print("Eine weitere Seite existiert; das Ergebnis ist ein begrenzter Ausschnitt.")
display(books)


Seiten: 2 | Beobachtungen: 6
Eindeutige Produkt-URLs: 5
Abbruchgrund: no_next_page


,title,price_gbp,detail_url,source_page
0,Datenpfade,19.9,http://127.0.0.1:57358/catalogue/datenpfade_00...,http://127.0.0.1:57358/page-1.html
1,HTML verstehen,29.0,http://127.0.0.1:57358/catalogue/html_verstehe...,http://127.0.0.1:57358/page-1.html
2,Kleine Pipelines,15.5,http://127.0.0.1:57358/catalogue/kleine_pipeli...,http://127.0.0.1:57358/page-1.html
3,Webdaten im Alltag,24.0,http://127.0.0.1:57358/catalogue/webdaten_im_a...,http://127.0.0.1:57358/page-2.html
4,Qualität vor Menge,9.9,http://127.0.0.1:57358/catalogue/qualitaet_vor...,http://127.0.0.1:57358/page-2.html
5,Datenpfade,19.9,http://127.0.0.1:57358/catalogue/datenpfade_00...,http://127.0.0.1:57358/page-2.html


## 4. Rohdaten, CSV und Herkunft speichern

Die CSV enthält alle Beobachtungen. Das Sample hat sechs Beobachtungen, aber fünf unterschiedliche Produkt-URLs: Eine Wiederholung bleibt hier bewusst sichtbar und wird im Task behandelt.

Ausgaben liegen getrennt nach `sample`/`live`; ein neuer Lauf ersetzt Dateien desselben Modus. Sample-Links enthalten einen temporären lokalen Port und sind nach Ende des Servers nicht mehr erreichbar. Detailseiten sind im Sample nicht enthalten.


In [6]:
output_dir = ROOT / "data/output/static_demo" / MODE
raw_dir = output_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
page_metadata = []
for number, page in enumerate(pages, start=1):
    filename = f"page-{number}.html"
    (raw_dir / filename).write_text(page["html"], encoding="utf-8")
    page_metadata.append({key: value for key, value in page.items() if key != "html"} | {"raw_file": "raw/" + filename})

books.to_csv(output_dir / "catalogue.csv", index=False, encoding="utf-8")
loaded = pd.read_csv(output_dir / "catalogue.csv", encoding="utf-8")
pd.testing.assert_frame_equal(books, loaded)
metadata = {
    "mode": MODE,
    "synthetic": not LIVE,
    "retrieved_at_utc": RETRIEVED_AT,
    "source": "Books to Scrape" if LIVE else "Synthetischer Lehrkatalog, kein Snapshot der Live-Website",
    "page_limit": MAX_PAGES,
    "pages_fetched": len(pages),
    "observations": len(books),
    "unique_detail_urls": int(books["detail_url"].nunique()),
    "stop_reason": stop_reason,
    "next_page_exists": remaining_url is not None,
    "pages": page_metadata,
}
(output_dir / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
if not LIVE:
    assert len(pages) == 2 and len(books) == 6 and books["detail_url"].nunique() == 5
print("CSV-Roundtrip erfolgreich.")
if not LIVE:
    print("Bekannte Sample-Anzahlen geprüft: 2 Seiten, 6 Beobachtungen, 5 Produkt-URLs.")
print("Ausgabe:", output_dir.relative_to(ROOT).as_posix())


CSV-Roundtrip erfolgreich.
Bekannte Sample-Anzahlen geprüft: 2 Seiten, 6 Beobachtungen, 5 Produkt-URLs.
Ausgabe: data/output/static_demo/sample


## Reflexion und Übergang zum Task

1. Welche Prüfung verhindert einen scheinbar erfolgreichen Export ohne Produktdaten?
2. Weshalb verwenden wir `response.url` als Basis für `urljoin`?
3. Was kannst Du über die Vollständigkeit sagen, wenn `stop_reason` gleich `page_limit` ist?
4. Welche zusätzlichen Felder und Regeln brauchst Du für eine Einkaufsliste nach Preis, Bewertung und Verfügbarkeit?

Bearbeite anschliessend [Task 1](../tasks/01_statischer_katalog/README.md). Dort entscheidest Du, wie Du Wiederholungen und widersprüchliche Werte behandelst.


## Quellen und Stand

Dokumentationsstand geprüft am **30.09.2026**: [Books to Scrape](https://books.toscrape.com/), [Requests Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/), [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) und [Python `urljoin`](https://docs.python.org/3/library/urllib.parse.html#urllib.parse.urljoin).

Books to Scrape ist ausdrücklich eine Übungswebsite; die dortigen Preise und Bewertungen haben keine reale Bedeutung. Unsere mitgelieferten Seiten enthalten eigene, vollständig synthetische Bücher. Sie bilden ausgewählte HTML-Muster nach und sind kein gespeicherter Auszug der Live-Website.
